# Tutorial 6: Parallel Training for Production Speed

## The Speed Problem

A single-environment PPO loop runs at ~1,600-2,000 fps on this environment. With 252-step episodes and 7 million total timesteps, that's:

```
7,000,000 / 1,800 fps ≈ 65 minutes
```

Not terrible, but not fast enough for serious hyperparameter search. And with a single environment, each policy update only sees ~8 episodes — poor coverage of the regime distribution.

## The Solution: SubprocVecEnv

Spectral-Env-Core was designed for parallel training from the ground up:

- **No matplotlib state** in the base environment → pickle-safe → multiprocessing works
- **Rendering is a wrapper** (`SpectralRenderWrapper`) applied only during evaluation
- **`deque`-based histories** with bounded memory → no memory leak across workers
- **Pre-computed price paths** in `reset()` → workers don't share state

With 8 workers: `65 min / 8 ≈ 8 minutes`. And each policy update now sees ~64 episodes (8 envs × 8 episodes per rollout), covering the full regime distribution.

---

In [ ]:
import time
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv, VecNormalize
from spectral_env_core import SpectralTradingEnv

In [ ]:
env_kwargs = {
    "num_assets": 3,
    "num_steps": 252,
    "initial_price": [150.0, 300.0, 50.0],
    "drift": [0.15, 0.20, 0.08],
    "volatility": [0.25, 0.30, 0.15],
    "garch_alpha": 0.08,
    "garch_beta": 0.85,
    "jump_intensity": 2.0,
    "jump_mean": -0.03,
    "jump_std": 0.06,
    "starting_cash": 100_000,
    "max_shares": 500,
    "max_trade_size": 50,
    "correlation": np.array([
        [1.0, 0.5, 0.2],
        [0.5, 1.0, 0.3],
        [0.2, 0.3, 1.0],
    ]),
}

## Benchmark: Single Env vs 8 Parallel Envs

In [ ]:
# Single env benchmark
single_env = make_vec_env(SpectralTradingEnv, n_envs=1, env_kwargs=env_kwargs)
single_env = VecNormalize(single_env, norm_obs=True, norm_reward=False, clip_obs=10.0)

model_single = PPO("MlpPolicy", single_env, n_steps=2048, batch_size=64, device="cpu", verbose=0)

start = time.time()
model_single.learn(total_timesteps=50_000)
single_time = time.time() - start
single_fps = 50_000 / single_time

print(f"Single env:  {single_time:.1f}s  ({single_fps:.0f} fps)")
single_env.close()

In [ ]:
# 8 parallel envs benchmark
parallel_env = make_vec_env(
    SpectralTradingEnv, n_envs=8, env_kwargs=env_kwargs, vec_env_cls=SubprocVecEnv
)
parallel_env = VecNormalize(parallel_env, norm_obs=True, norm_reward=False, clip_obs=10.0)

model_parallel = PPO("MlpPolicy", parallel_env, n_steps=2048, batch_size=64, device="cpu", verbose=0)

start = time.time()
model_parallel.learn(total_timesteps=50_000)
parallel_time = time.time() - start
parallel_fps = 50_000 / parallel_time

print(f"8 parallel envs: {parallel_time:.1f}s  ({parallel_fps:.0f} fps)")
print(f"\nSpeedup: {single_time / parallel_time:.1f}×")
parallel_env.close()

## Why Parallel Training Matters Beyond Speed

Speed is the obvious benefit. But there's a deeper one: **gradient quality**.

With 1 environment and `n_steps=2048`, each policy update sees ~8 complete episodes — all from the same random seed of regime sequences and GARCH realisations. The policy gradient is biased toward whatever market character those 8 episodes happened to produce.

With 8 environments, the same rollout covers ~64 episodes — each with independently drawn regimes, GARCH paths, jump events, and AR(1) coefficients. The gradient estimate is much less noisy and covers the full distribution of market characters at every update.

This is especially important for regime switching: with 1 env, a single rollout might be entirely "bull" or entirely "bear." With 8 envs, you're guaranteed both regimes appear in every batch.

---

## The Full Production Pipeline

The included `train.py` puts it all together:

```python
# train.py highlights:

# 1. Parallel training envs
train_vec = make_vec_env(SpectralTradingEnv, n_envs=8, vec_env_cls=SubprocVecEnv, ...)
train_vec = VecNormalize(train_vec, norm_obs=True, ...)

# 2. Custom feature extractor (shared encoder across assets)
policy_kwargs = dict(
    features_extractor_class=SpectralExtractor,
    net_arch=dict(pi=[64, 64], vf=[128, 128, 64]),  # deeper critic
)

# 3. Diagnostics callback — tracks explained variance, friction, action stats
diag_callback = DiagnosticsCallback(...)

# 4. Entropy coefficient decay — explore early, exploit late
ent_callback = EntropyCoefficientCallback(start=0.005, end=0.001)

model.learn(total_timesteps=7_000_000, callback=[ent_callback, diag_callback, eval_callback])
```

Run it:
```bash
python train.py
```

Monitor with TensorBoard:
```bash
tensorboard --logdir ./logs
```

Analyse with the diagnostic dashboard:
```bash
jupyter notebook logs/examine.ipynb
```

---

## Design Decision: Why Rendering is Decoupled

Most gym environments include rendering logic directly. This creates a problem: `matplotlib.figure.Figure` objects can't be pickled, which means `SubprocVecEnv` (which spawns worker processes) crashes on environment creation.

Spectral-Env-Core solves this by making the base `SpectralTradingEnv` completely rendering-free. All matplotlib logic lives in `SpectralRenderWrapper` — a Gymnasium wrapper you apply only during human evaluation:

```python
# Training: no wrapper, pickle-safe
train_env = make_vec_env(SpectralTradingEnv, vec_env_cls=SubprocVecEnv, ...)

# Evaluation: add wrapper for visualisation
from spectral_env_core import SpectralRenderWrapper
eval_env = SpectralRenderWrapper(SpectralTradingEnv(...))
```

This is the same pattern used by MuJoCo and Atari environments in Gymnasium — it's the correct architecture for production RL training.